# 05 · เพิ่มผู้นำหนึ่ง agent (เงื่อนไข D, ระยะที่ 6, RQ3)

ผู้นำ LLM ตัดสินใจ **ทุก 7 วัน** จากรายงานผู้ป่วยเฉลี่ย 7 วัน แนวโน้ม และสัดส่วนคนไปทำงาน แล้วเลือกระดับ `open / limited / strict`
ระดับนโยบายมีผล 3 ทางเท่านั้น: $q_{policy} = 1 - k\,d$, ข้อความมาตรการที่ประชาชนเห็น และ action ที่ถูกห้าม (`strict` ห้าม `visit_family`)
ผู้นำ **ไม่** แก้สถานะ SEIR โดยตรง

ทำ ablation: D เทียบกับ C ด้วยโมเดล, persona และ seed เดียวกัน

In [ ]:
# ---- Setup (same cell in every notebook) ----
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/ThaiGABM")   # where you uploaded the abm folder
    IN_COLAB = True
except ImportError:                                   # running locally (notebooks 00 and 01)
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False

assert (ROOT / "gabm").exists(), f"gabm/ not found in {ROOT}: upload the abm folder to MyDrive/ThaiGABM"
sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-colab.txt")], check=True)

from gabm import load_config
cfg = load_config(ROOT / "config" / "base.yaml")
FIG = ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| Colab:", IN_COLAB)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "no GPU")

In [ ]:
# Hugging Face token: needed only for gated models (llama8b). Add it in Colab: key icon (Secrets) -> HF_TOKEN
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF_TOKEN loaded")
    except Exception:
        print("No HF_TOKEN secret: gated models will fail to download")

In [ ]:
MODEL_KEY = "qwen7b"              # @param ["typhoon8b", "qwen7b", "llama8b", "openthaigpt7b", "seallm7b", "mistral7b", "qwen1_5b"]
BACKEND = cfg["llm"]["backend"]
SEEDS = cfg["sim"]["seeds"]
DAYS = 180                        # must match the horizon used for condition C in notebook 04
RUN_LLM = True                    # False: only run the analysis cells (no GPU)
FORCE = False                     # run even if the prototype criteria below are not met

## เกณฑ์ก่อนเพิ่มผู้นำ (หัวข้อ 12)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from gabm.sim import run_simulation, contact_model_from_cfg
from gabm.llm import load_backend
from gabm.analysis import load_runs, load_daily, plot_epidemic, plot_action_shares

cfg["sim"]["days"] = DAYS   # here, not in the parameters cell, so a papermill override of DAYS also reaches cfg
runs = load_runs(ROOT)
C = runs[(runs.condition == "C") & (runs.model_key == MODEL_KEY) & (runs.tag == "") & (runs.days == DAYS)]
checks = {
    "SEIR baseline run exists (A)": (runs.condition == "A").any(),
    f"C runs for {MODEL_KEY} on all seeds": set(SEEDS) <= set(C.seed),
    "fallback rate < 5%": len(C) > 0 and C.fallback_rate.max() < 0.05,
    "feasibility report exists": (ROOT / "outputs" / "feasibility" / "feasibility.csv").exists(),
}
for k, v in checks.items():
    print(("✓" if v else "✗"), k)
assert all(checks.values()) or FORCE, "Criteria not met: finish notebooks 01-03 first, or set FORCE = True"

In [ ]:
personas = pd.read_csv(ROOT / cfg["personas"]["file"])
cm = contact_model_from_cfg(cfg, ROOT)
if RUN_LLM:
    backend = load_backend(cfg["llm"], MODEL_KEY, BACKEND)
    for s in SEEDS:
        run_simulation(cfg, "D", seed=s, personas=personas, backend=backend, model_key=MODEL_KEY, cm=cm, root=ROOT)

## Ablation: D เทียบกับ C

In [ ]:
runs = load_runs(ROOT)
CD = runs[runs.condition.isin(["C", "D"]) & (runs.model_key == MODEL_KEY) & (runs.tag == "") & runs.seed.isin(SEEDS) & (runs.days == DAYS)]
metrics = ["attack_rate_window", "peak_I", "peak_day", "mean_q_out", "mean_beta_eff", "fallback_rate"]
display(CD.groupby("condition")[metrics].agg(["mean", "std"]).round(4))
paired = CD.pivot_table(index="seed", columns="condition", values="attack_rate_window")
print("paired difference D - C in attack rate by seed:", (paired["D"] - paired["C"]).round(4).to_dict())

daily = load_daily(list(CD.run_id), ROOT)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
plot_epidemic(daily, ax=axes[0], y="I"); axes[0].set_title("Infectious I")
plot_epidemic(daily, ax=axes[1], y="beta_eff"); axes[1].set_title("beta_eff")
plot_action_shares(daily[daily.condition == "D"], ax=axes[2], title="Condition D action shares")
plt.tight_layout(); plt.savefig(FIG / f"05_leader_{MODEL_KEY}.png", dpi=150); plt.show()

In [ ]:
Druns = CD[CD.condition == "D"]
leader = pd.concat([pd.read_json(ROOT / "outputs" / "runs" / rid / "leader.jsonl", lines=True).assign(seed=sd)
                    for rid, sd in zip(Druns.run_id, Druns.seed)], ignore_index=True)
display(leader[["seed", "day", "choice", "reason", "valid", "retried", "fallback"]])